✅ 1. Time-lag Feature Importance 분석 (Lagged Outcome Modeling)
📌 핵심 아이디어
- 동일한 입력 X (정책 포함, 2020년 데이터)를 사용해 서로 다른 시점의 Y (2020, 2021, 2022)를 예측하는 3개의 모델을 훈련
- 정책 관련 feature의 중요도를 연도별로 비교

✅ 기대 해석:
- 'policy_var'의 SHAP 중요도가 가장 높은 연도가 → 정책 효과가 가장 강하게 나타난 시점.

✅ 2. Causal Inference 기반 머신러닝 (e.g., Causal Forest)
📌 핵심 아이디어: 정책의 처리 여부 (treated vs untreated) 를 기준으로, 정책 도입이 결과(Y)에 미치는 인과적 효과 (ATE, CATE) 를 시점별로 추정


In [ ]:
#📊 예제 (with LightGBM + SHAP)

import lightgbm as lgb
import shap
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 예시 데이터
X_2020 = pd.read_csv("X_2020.csv")  # 정책 포함된 2020년 변수
Y_2020 = pd.read_csv("Y_2020.csv")
Y_2021 = pd.read_csv("Y_2021.csv")
Y_2022 = pd.read_csv("Y_2022.csv")

# 함수: 모델 훈련 + SHAP 분석
def train_and_analyze(X, Y):
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
    model = lgb.LGBMClassifier()
    model.fit(X_train, y_train)

    # SHAP 값 계산
    explainer = shap.Explainer(model)
    shap_values = explainer(X_test)

    # 중요도 평균 계산
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': shap_values.values.mean(axis=0)
    }).sort_values(by='importance', ascending=False)

    return model, importance

model_2020, imp_2020 = train_and_analyze(X_2020, Y_2020.values.ravel())
model_2021, imp_2021 = train_and_analyze(X_2020, Y_2021.values.ravel())
model_2022, imp_2022 = train_and_analyze(X_2020, Y_2022.values.ravel())

# 예시 출력
print("Policy 관련 변수 중요도 (연도별)")
print("2020:", imp_2020.query("feature == 'policy_var'")['importance'].values)
print("2021:", imp_2021.query("feature == 'policy_var'")['importance'].values)
print("2022:", imp_2022.query("feature == 'policy_var'")['importance'].values)

✅ 2. Causal Inference 기반 머신러닝 (e.g., Causal Forest)
📌 핵심 아이디어: 정책의 처리 여부 (treated vs untreated) 를 기준으로, 정책 도입이 결과(Y)에 미치는 인과적 효과 (ATE, CATE) 를 시점별로 추정

📊 예제 (with EconML의 CausalForestDML)

✅ 기대 해석:
- ATE_2021 > ATE_2020 → 정책 효과가 2021년에 본격적으로 나타났다고 해석 가능
- CATE도 구하면 어떤 그룹에서 효과가 컸는지 파악 가능

In [ ]:
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
import numpy as np

# 데이터 구성
# X_2020: 2020년 정책 및 조직 특성
# T: treatment (정책을 적용받은 조직이면 1, 아니면 0)
# Y_t: 각 해의 결과

T = X_2020['policy_applied']  # 1 or 0
X = X_2020.drop(columns=['policy_applied'])

# 모델 정의
model_2021 = CausalForestDML(
    model_t=LassoCV(),
    model_y=RandomForestRegressor(),
    discrete_treatment=True,
    random_state=42
)

model_2021.fit(Y_2021.values.ravel(), T, X=X, W=None)
te_2021 = model_2021.effect(X)
ate_2021 = np.mean(te_2021)

print("2021년의 ATE (Average Treatment Effect):", ate_2021)

# 동일한 방식으로 Y_2020, Y_2022도 수행 가능


✅ 두 방법 비교 요약
- 방법                         /	해석 대상            / 결과값        /	장점                     /	활용 예
- Time-lag Feature Importance /	모델 feature 중요도	/ SHAP value  /	직관적, 해석 쉬움          / 정책 효과 시점 추정
- Causal Forest               /	인과 효과	        / ATE & CATE  /	인과성 강조, 논문 설득력↑   / 정책 효과 자체 추정
